# Training Your Own RL Agent on SupplyMind

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShAuRyA-Noodle/Sleep-Token/blob/main/notebooks/02_training_your_own_agent.ipynb)

This notebook walks through training a PPO agent on the SupplyMind environment with full hyperparameter explanations.

In [ ]:
# Install dependencies
!pip install -q gymnasium==0.29.1 stable-baselines3==2.2.1 sb3-contrib==2.2.1 torch networkx numpy pydantic fastapi

In [ ]:
import gymnasium as gym
import torch
import numpy as np
import rl

from sb3_contrib import MaskablePPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Create vectorized environment
def make_env(seed, rank):
    def _init():
        env = gym.make('SupplyMind-Easy-v1')
        env.reset(seed=seed + rank)
        return env
    return _init

# 4 parallel environments (increase to 32 on GPU)
n_envs = 4
env = DummyVecEnv([make_env(42, i) for i in range(n_envs)])
env = VecNormalize(env, norm_obs=True, norm_reward=True)
print(f'Vectorized env with {n_envs} parallel workers')

In [ ]:
# Configure PPO
model = MaskablePPO(
    'MlpPolicy',
    env,
    n_steps=512,         # Steps per env before update (lower for Colab)
    batch_size=128,      # Mini-batch size for gradient updates
    learning_rate=3e-4,  # Adam learning rate
    gamma=0.99,          # Discount factor (long-term planning)
    gae_lambda=0.95,     # GAE smoothing (bias-variance tradeoff)
    clip_range=0.2,      # PPO clipping (prevents too-large updates)
    ent_coef=0.01,       # Entropy bonus (encourages exploration)
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=1,
)
print(f'Training on {model.device}')

In [ ]:
# Train! (50K steps for demo, use 2M for real training)
model.learn(total_timesteps=50_000, progress_bar=True)

In [ ]:
# Evaluate
eval_env = gym.make('SupplyMind-Easy-v1')
rewards = []

for ep in range(10):
    obs, info = eval_env.reset(seed=ep)
    total_r = 0
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = eval_env.step(action)
        total_r += r
        if term or trunc:
            break
    rewards.append(total_r)

print(f'Mean reward: {np.mean(rewards):.4f} +/- {np.std(rewards):.4f}')